# Benchmarks

## Initialize

In [1]:
import os
import math
import pathlib
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from IPython.display import clear_output

import warnings
from lifelines.utils import CensoringType
from lifelines.utils import concordance_index

In [2]:
node = !hostname
if "sc" in node[0]:
    base_path = "/sc-projects/sc-proj-ukb-cvd"
else: 
    base_path = "/data/analysis/ag-reils/ag-reils-shared/cardioRS"
print(base_path)

project_label = "22_medical_records"
project_path = f"{base_path}/results/projects/{project_label}"
figure_path = f"{project_path}/figures"
output_path = f"{project_path}/data"

experiment = 220413
experiment_path = f"{output_path}/{experiment}"
pathlib.Path(experiment_path).mkdir(parents=True, exist_ok=True)

/sc-projects/sc-proj-ukb-cvd


In [4]:
model_path = f"{experiment_path}/coxph/models"
model_list =  !ls $model_path

In [ ]:
!ls -lt $model_path

In [7]:
endpoints_md = pd.read_csv(f"{experiment_path}/endpoints.csv")
endpoints = sorted(endpoints_md.endpoint.to_list())

In [8]:
partitions = [p for p in range(0, 22)]
splits = ["train", "valid", "test"]

In [9]:
endpoint_defs = pd.read_feather(f"{output_path}/phecode_defs_220306.feather").query("endpoint==@endpoints").sort_values("endpoint").set_index("endpoint")

In [10]:
eligable_eids = pd.read_feather(f"{output_path}/eligable_eids_220414.feather")
eids_dict = eligable_eids.set_index("endpoint")["eid_list"].to_dict()

In [11]:
%env MKL_NUM_THREADS=4
%env NUMEXPR_NUM_THREADS=4
%env OMP_NUM_THREADS=4

env: MKL_NUM_THREADS=4
env: NUMEXPR_NUM_THREADS=4
env: OMP_NUM_THREADS=4


In [18]:
ray.shutdown()

NameError: name 'ray' is not defined

In [12]:
import ray

ray.init(num_cpus=24)#, dashboard_port=24762, dashboard_host="0.0.0.0", include_dashboard=True)#, webui_url="0.0.0.0"))

2022-04-21 12:12:34,681	INFO services.py:1412 -- View the Ray dashboard at http://127.0.0.1:8266


{'node_ip_address': '10.32.105.13',
 'raylet_ip_address': '10.32.105.13',
 'redis_address': None,
 'object_store_address': '/tmp/ray/session_2022-04-21_12-12-28_789006_262444/sockets/plasma_store',
 'raylet_socket_name': '/tmp/ray/session_2022-04-21_12-12-28_789006_262444/sockets/raylet',
 'webui_url': '127.0.0.1:8266',
 'session_dir': '/tmp/ray/session_2022-04-21_12-12-28_789006_262444',
 'metrics_export_port': 54910,
 'gcs_address': '10.32.105.13:45339',
 'address': '10.32.105.13:45339',
 'node_id': '1b33f67968a8d69dcd30a4eca7fcaf2bc90a6206b49a42d8ba4c27e5'}

# Predict COX

In [13]:
in_path = pathlib.Path(f"{experiment_path}/coxph/input")
model_path = f"{experiment_path}/coxph/models"

out_path = f"{experiment_path}/coxph/predictions"
pathlib.Path(out_path).mkdir(parents=True, exist_ok=True)

In [14]:
models = ['Identity(Records)+MLP']

In [15]:
AgeSex = ["age_at_recruitment_f21022_0_0", "sex_f31_0_0"]

In [16]:
from lifelines import CoxPHFitter
from lifelines.exceptions import ConvergenceError
import zstandard
import pickle
import os

def get_features(endpoint):
    features = {
        'Identity(Records)+MLP': {
            "Age+Sex": AgeSex,
            "MedicalHistory": [endpoint],
            "Age+Sex+MedicalHistory": AgeSex + [endpoint],
            "Age+Sex+MedicalHistory+I(Age*MH)": AgeSex + [endpoint]
            }
    }
    return features

def get_test_data(in_path, partition, models, mapping):
    train_data = {model: pd.read_feather(f"{in_path}/{model}/{partition}/test.feather").set_index("eid").replace(mapping)for model in models}
    return train_data
            
def load_pickle(fp):
    with open(fp, "rb") as fh:
        dctx = zstandard.ZstdDecompressor()
        with dctx.stream_reader(fh) as decompressor:
            data = pickle.loads(decompressor.read())
    return data

def predict_cox(cph, data_endpoint, endpoint, feature_set, partition, pred_path):
    times = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
    time_cols = {t: f"Ft_{t}" for t in times}
    
    if feature_set=="Age+Sex+MedicalHistory+I(Age*MH)":
        data_endpoint.columns = [c.replace("-", "") for c in data_endpoint.columns]
    
    surv_test = 1-cph.predict_survival_function(data_endpoint, times=times) 
    temp_pred = data_endpoint.reset_index()[["eid"]].assign(endpoint=endpoint, features=feature_set, partition=partition)
    for t, col in time_cols.items(): temp_pred[col] = surv_test.T[t].to_list()
    
    temp_pred.to_feather(f"{out_path}/{endpoint}_{feature_set}_{partition}.feather")

@ray.remote
def predict_endpoint(data_partition, eids_dict, endpoint, partition, models, features, model_path, out_path):
    eids_incl = eids_dict[endpoint].tolist()
    results = []
    for model in models:
        data_model = data_partition[model]
        for feature_set, covariates in features[model].items():
            identifier = f"{endpoint}_{feature_set}_{partition}"
            pred_path = f"{out_path}/{identifier}.feather"
            if not os.path.isfile(pred_path):
                try:
                    cph = load_pickle(f"{model_path}/{identifier}.p")
                    data_endpoint = data_model[data_model.index.isin(eids_incl)]
                    predict_cox(cph, data_endpoint, endpoint, feature_set, partition, pred_path)
                except:
                    print(f"{identifier} not available")
    return True

In [17]:
mapping = {"sex_f31_0_0": {"Female":0, "Male":1}}

ray_eids = ray.put(eids_dict)
for partition in tqdm(partitions):
    try:
        del ray_partition
    except:
        print("Ray object not yet initialised")
    ray_partition = ray.put(get_test_data(in_path, partition, models, mapping))
    progress = []
    for endpoint in endpoints:
        features = get_features(endpoint)
        progress.append(predict_endpoint.remote(ray_partition, ray_eids, endpoint, partition, models, features, model_path, out_path))
    [ray.get(s) for s in tqdm(progress)]

  0%|          | 0/22 [00:00<?, ?it/s]

Ray object not yet initialised


  0%|          | 0/1575 [00:00<?, ?it/s]

  0%|          | 0/1575 [00:00<?, ?it/s]

  0%|          | 0/1575 [00:00<?, ?it/s]

...

  0%|          | 0/1575 [00:00<?, ?it/s]

  0%|          | 0/1575 [00:00<?, ?it/s]

  0%|          | 0/1575 [00:00<?, ?it/s]

  0%|          | 0/1575 [00:00<?, ?it/s]

  0%|          | 0/1575 [00:00<?, ?it/s]